<a href="https://colab.research.google.com/github/jaysulk/GENERIC-FNO/blob/main/GENERIC_FNO_Figures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
# ============================================================================
# GENERIC-FNO -- SINGLE SELF-CONTAINED NOTEBOOK SCRIPT (no external files).
# Part A: the model classes + PDE generators (verbatim, so saved checkpoints
#         load exactly).  Part B: all paper figures.
# Paste this as ONE cell and run.  Then:
#     main()                                  # all 8 figures
#     fig_interpretability(models={'heat': m_heat, 'burgers': m_b, ...})  # in-memory
# fig_interpretability uses GENERIC_FNO2d defined right here; weights come from
# either in-memory `models=` you pass, or the nx128 checkpoints on BASE.
# ============================================================================

# ==================== PART A: MODEL + GENERATORS (verbatim) ==================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pickle
import time
import math
from collections import defaultdict


# ============================================================================
# Data Generation — 2D PDEs (spectral)
# ============================================================================

def _random_field_2d(nx, ny, max_mode, n_modes, amp_scale=0.5, device='cpu'):
    """Build a random smooth 2D field as a sum of sine modes."""
    x = torch.linspace(0, 2*math.pi, nx+1, device=device)[:-1]
    y = torch.linspace(0, 2*math.pi, ny+1, device=device)[:-1]
    X, Y = torch.meshgrid(x, y, indexing='ij')
    u = torch.zeros(nx, ny, device=device)
    for _ in range(n_modes):
        kx = torch.randint(1, max_mode, (1,)).item()
        ky = torch.randint(1, max_mode, (1,)).item()
        amp = torch.randn(1).item() * amp_scale
        phase = torch.rand(1).item() * 2 * math.pi
        u += amp * torch.sin(kx * X + ky * Y + phase)
    return u


def generate_heat_data_2d(n_samples=150, nx=128, nt=15, dt=0.005, nu=0.02, device='cpu'):
    """2D heat: du/dt = nu*(uxx+uyy). Purely dissipative. Exact in spectral space."""
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    k_sq = KX**2 + KY**2
    decay = torch.exp(-nu * k_sq * dt)

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(3, 7, (1,)).item()
        u0 = _random_field_2d(nx, nx, nx//8, n_modes, device=device)
        u_hat = torch.fft.rfft2(u0)
        traj = [u0.clone()]
        for t in range(nt):
            u_hat = u_hat * decay
            traj.append(torch.fft.irfft2(u_hat, s=(nx, nx)))
        traj = torch.stack(traj, dim=0)  # (nt+1, nx, nx)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'heat'


def generate_wave_data_2d(n_samples=150, nx=128, nt=15, dt=0.005, c=1.0, device='cpu'):
    """2D wave: u_tt = c²(uxx+uyy). Reversible. Track u-component, exact spectral rotation."""
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    kmag = torch.sqrt(KX**2 + KY**2)
    omega = c * kmag

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(3, 7, (1,)).item()
        u0 = _random_field_2d(nx, nx, nx//8, n_modes, device=device)
        v0 = _random_field_2d(nx, nx, nx//8, n_modes, amp_scale=0.3, device=device)
        u_hat = torch.fft.rfft2(u0)
        v_hat = torch.fft.rfft2(v0)
        traj = [u0.clone()]
        for t in range(nt):
            cos_w = torch.cos(omega * dt)
            sin_w = torch.sin(omega * dt)
            # rotate (u, v) preserving energy; guard omega=0 mode
            safe_omega = torch.where(omega > 1e-8, omega, torch.ones_like(omega))
            u_new = cos_w * u_hat + (sin_w / safe_omega) * v_hat
            v_new = -safe_omega * sin_w * u_hat + cos_w * v_hat
            u_hat, v_hat = u_new, v_new
            traj.append(torch.fft.irfft2(u_hat, s=(nx, nx)))
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'wave'


def generate_advection_data_2d(n_samples=150, nx=128, nt=15, dt=0.005, c=1.0,
                               max_mode=6, device='cpu'):
    """2D linear advection: u_t + c(u_x+u_y) = 0. Reversible, Markovian in u,
    conserves 0.5<u^2> exactly. Clean fully-observed reversible scalar test
    => a thermodynamically-consistent operator should drive M -> 0.
    Band-limited to max_mode (fixed, NOT nx//8) so the content stays within the
    operator's mode range and the per-step phase rotation is small -- otherwise
    high-k transport aliases and even a plain FNO fails (the operators only span
    the lowest modes_op modes)."""
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    phase = torch.exp(-1j * c * (KX + KY) * dt)
    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(3, 7, (1,)).item()
        u0 = _random_field_2d(nx, nx, max_mode, n_modes, device=device)
        u_hat = torch.fft.rfft2(u0)
        traj = [u0.clone()]
        for t in range(nt):
            u_hat = u_hat * phase
            traj.append(torch.fft.irfft2(u_hat, s=(nx, nx)))
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'advection'


def generate_burgers_data_2d(n_samples=150, nx=128, nt=15, dt=0.002, nu=0.02, device='cpu'):
    """2D scalar Burgers: u_t + u(u_x+u_y) = nu*(uxx+uyy). Mixed rev+diss.
    Semi-implicit: diffusion in spectral, advection explicit."""
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    k_sq = KX**2 + KY**2

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 5, (1,)).item()
        u = _random_field_2d(nx, nx, 5, n_modes, amp_scale=0.3, device=device)
        traj = [u.clone()]
        for t in range(nt):
            u_hat = torch.fft.rfft2(u)
            # implicit diffusion
            u_hat = u_hat / (1 + nu * k_sq * dt)
            u = torch.fft.irfft2(u_hat, s=(nx, nx))
            # explicit advection (spectral derivatives)
            ux = torch.fft.irfft2(1j * KX * torch.fft.rfft2(u), s=(nx, nx))
            uy = torch.fft.irfft2(1j * KY * torch.fft.rfft2(u), s=(nx, nx))
            u = u - dt * u * (ux + uy)
            traj.append(u.clone())
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'burgers'


# ============================================================================
# Building Blocks — 2D
# ============================================================================

class SpectralConv2d(nn.Module):
    """Standard FNO spectral convolution (2D). Two corners for +/- kx."""
    def __init__(self, in_ch, out_ch, modes1, modes2):
        super().__init__()
        self.modes1 = modes1  # kx modes (keep both +/- corners)
        self.modes2 = modes2  # ky modes (non-negative only, rfft)
        scale = 1.0 / (in_ch * out_ch)
        self.W1 = nn.Parameter(scale * torch.randn(out_ch, in_ch, modes1, modes2, dtype=torch.cfloat))
        self.W2 = nn.Parameter(scale * torch.randn(out_ch, in_ch, modes1, modes2, dtype=torch.cfloat))

    def forward(self, x):
        B, C, H, W = x.shape
        x_hat = torch.fft.rfft2(x, dim=(-2, -1))  # (B, C, H, W//2+1)
        out_hat = torch.zeros(B, self.W1.shape[0], H, W // 2 + 1,
                              dtype=torch.cfloat, device=x.device)
        m1 = min(self.modes1, H // 2)
        m2 = min(self.modes2, W // 2 + 1)
        # top-left corner (positive kx)
        out_hat[:, :, :m1, :m2] = torch.einsum(
            'bixy,oixy->boxy', x_hat[:, :, :m1, :m2], self.W1[:, :, :m1, :m2])
        # bottom-left corner (negative kx)
        out_hat[:, :, -m1:, :m2] = torch.einsum(
            'bixy,oixy->boxy', x_hat[:, :, -m1:, :m2], self.W2[:, :, :m1, :m2])
        return torch.fft.irfft2(out_hat, s=(H, W))


class AAGELU(nn.Module):
    """Anti-aliased GELU. A pointwise nonlinearity injects high-frequency harmonics
    that ALIAS on a coarse grid, which is the main reason FNO-style nets are only
    approximately resolution-invariant. We upsample by `factor` (band-limited, via
    FFT zero-pad), apply GELU on the finer grid, then downsample (FFT truncate),
    which suppresses the aliased content. Operates on whatever (H,W) it receives,
    so it stays resolution-agnostic. Assumes even H,W (true for our grids)."""
    def __init__(self, factor=2):
        super().__init__()
        self.f = factor

    def forward(self, x):
        f = self.f
        if f == 1:
            return F.gelu(x)
        B, C, H, W = x.shape
        Xs = torch.fft.fftshift(torch.fft.fft2(x, dim=(-2, -1)), dim=(-2, -1))
        Hf, Wf = H * f, W * f
        ph, pw = (Hf - H) // 2, (Wf - W) // 2
        up = F.pad(Xs, (pw, Wf - W - pw, ph, Hf - H - ph))
        x_up = torch.fft.ifft2(torch.fft.ifftshift(up, dim=(-2, -1)), dim=(-2, -1)).real * (f * f)
        x_up = F.gelu(x_up)
        Ys = torch.fft.fftshift(torch.fft.fft2(x_up, dim=(-2, -1)), dim=(-2, -1))
        crop = Ys[..., ph:ph + H, pw:pw + W]
        return torch.fft.ifft2(torch.fft.ifftshift(crop, dim=(-2, -1)), dim=(-2, -1)).real / (f * f)


def _act(antialias):
    return AAGELU(2) if antialias else nn.GELU()


class FNO_Block2d(nn.Module):
    def __init__(self, width, modes1, modes2, antialias=False):
        super().__init__()
        self.conv = SpectralConv2d(width, width, modes1, modes2)
        self.skip = nn.Conv2d(width, width, 1)
        self.norm = nn.InstanceNorm2d(width)
        self.act = _act(antialias)

    def forward(self, x):
        return self.act(self.norm(self.conv(x) + self.skip(x)))


class FNO_Backbone2d(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, width=32, modes=16, n_layers=4, antialias=False):
        super().__init__()
        self.lift = nn.Conv2d(in_ch, width, 1)
        self.blocks = nn.ModuleList([FNO_Block2d(width, modes, modes, antialias=antialias)
                                     for _ in range(n_layers)])
        self.proj = nn.Sequential(
            nn.Conv2d(width, width, 1),
            _act(antialias),
            nn.Conv2d(width, out_ch, 1)
        )

    def forward(self, x):
        x = self.lift(x)
        for block in self.blocks:
            x = block(x)
        return self.proj(x)


class FunctionalNet2d(nn.Module):
    """FNO2d backbone → scalar functional F[u]. u:(B,1,H,W) → (B,)."""
    def __init__(self, width=24, modes=12, n_layers=3, antialias=False):
        super().__init__()
        self.backbone = FNO_Backbone2d(in_ch=1, out_ch=1, width=width,
                                        modes=modes, n_layers=n_layers, antialias=antialias)
        self.head = nn.Sequential(
            nn.Linear(1, 16),
            nn.GELU(),
            nn.Linear(16, 1)
        )

    def forward(self, u):
        density = self.backbone(u)              # (B,1,H,W)
        integral = density.mean(dim=(-1, -2))   # (B,1) — spatial average ∝ integral
        return self.head(integral).squeeze(-1)  # (B,)


# ============================================================================
# Model 1: Vanilla FNO (2D, residual)
# ============================================================================

class VanillaFNO2d(nn.Module):
    def __init__(self, width=32, modes=16, n_layers=4):
        super().__init__()
        self.backbone = FNO_Backbone2d(in_ch=1, out_ch=1, width=width,
                                        modes=modes, n_layers=n_layers)

    def forward(self, u):
        return u + self.backbone(u)

    def predict_with_info(self, u):
        return self.forward(u), {}


# ============================================================================
# Model 2: EP-FNO (2D, energy penalty)
# ============================================================================

class EP_FNO2d(nn.Module):
    def __init__(self, width=32, modes=16, n_layers=4):
        super().__init__()
        self.backbone = FNO_Backbone2d(in_ch=1, out_ch=1, width=width,
                                        modes=modes, n_layers=n_layers)

    def forward(self, u):
        return u + self.backbone(u)

    def predict_with_info(self, u):
        u_next = self.forward(u)
        E_in = 0.5 * (u**2).mean(dim=(-1, -2)).mean(dim=-1)
        E_out = 0.5 * (u_next**2).mean(dim=(-1, -2)).mean(dim=-1)
        return u_next, {'dE': E_out - E_in}

    def energy_penalty(self, info, pde_type):
        dE = info['dE']
        if pde_type in ('heat', 'burgers'):
            return (F.relu(dE)**2).mean()
        elif pde_type in ('wave', 'advection'):
            return (dE**2).mean()
        return torch.tensor(0.0, device=dE.device)


# ============================================================================
# Model 3: GENERIC-FNO (2D)
# ============================================================================

class GENERIC_FNO2d(nn.Module):
    """
    du/dt = L·δE/δu + M·δS/δu  with hard projection.
    L(kx,ky) = i·a  (anti-Hermitian diagonal), M(kx,ky) = |b|² (PSD diagonal).
    Operators are defined on the lowest (modes_op) modes in each direction,
    using two corners for +/- kx (rfft2 layout) → resolution invariant.
    """
    def __init__(self, nx=128, width_func=24, modes_func=12, n_layers_func=3,
                 modes_op=16, residual_gate_init=-3.0, l2_vargrad=False,
                 use_residual=True, degeneracy_construction=True,
                 antialias=False, integrator='euler'):
        super().__init__()
        self.nx = nx
        self.modes_op = modes_op
        # degeneracy_construction (DEFAULT, the thermodynamically-consistent model):
        #   build L = (I-P_S) D_L (I-P_S) and M = (I-P_E) D_M (I-P_E), where P_E,P_S
        #   are rank-1 projections onto delta E/delta u, delta S/delta u and D_L=i*a,
        #   D_M=|b|^2 are diagonal Fourier multipliers. Then L dS = 0 and M dE = 0
        #   EXACTLY, so energy is conserved (dE/dt=0) and entropy is produced
        #   (dS/dt=<dS,M dS> >= 0) by construction in ANY dimension -- no energy
        #   projection, no entropy correction, no free residual. A reversible PDE
        #   (wave) is forced to learn M->0 because dissipation can no longer hide
        #   behind a projection. Set False for the legacy projection+correction path
        #   (kept only for ablation; it does NOT specialize -- M and L are
        #   interchangeable under the projection, so everything routes through M).
        # l2_vargrad: use the L2 variational derivative (N/|Omega|) grad_u E. Affects
        #   only the operator-output SCALE (projections are scale-invariant ratios);
        #   harmless either way under the construction. Default off.
        # use_residual: only meaningful in the legacy path; a free residual would
        #   break the thermodynamic guarantee, so it is ignored when
        #   degeneracy_construction=True.
        self.degeneracy_construction = degeneracy_construction
        self.l2_vargrad = l2_vargrad
        self.use_residual = use_residual
        self.integrator = integrator   # 'euler' (default) or 'rk4' (norm-preserving)
        m1 = min(modes_op, nx // 2)
        m2 = min(modes_op, nx // 2 + 1)
        self.m1, self.m2 = m1, m2

        self.E_net = FunctionalNet2d(width=width_func, modes=modes_func,
                                     n_layers=n_layers_func, antialias=antialias)
        self.S_net = FunctionalNet2d(width=width_func, modes=modes_func,
                                     n_layers=n_layers_func, antialias=antialias)

        # L: anti-Hermitian diagonal, two kx corners
        self.a_pos = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.a_neg = nn.Parameter(0.3 * torch.randn(m1, m2))
        # M: PSD diagonal, two kx corners (parameterized as |b|²)
        self.b_pos_r = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.b_pos_i = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.b_neg_r = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.b_neg_i = nn.Parameter(0.3 * torch.randn(m1, m2))

        # Small gated residual for high-freq content
        self.residual = nn.Sequential(
            nn.Conv2d(1, 16, 1),
            nn.GELU(),
            nn.Conv2d(16, 1, 1)
        )
        self.residual_gate = nn.Parameter(torch.tensor(float(residual_gate_init)))

    def _apply_operators(self, dEdu_hat, dSdu_hat, H, W):
        """Apply diagonal L and M in 2D Fourier space (two kx corners)."""
        m1, m2 = self.m1, self.m2
        rev_hat = torch.zeros_like(dEdu_hat)
        diss_hat = torch.zeros_like(dSdu_hat)

        # L = i*a  (reversible)
        rev_hat[:, :, :m1, :m2] = 1j * self.a_pos * dEdu_hat[:, :, :m1, :m2]
        rev_hat[:, :, -m1:, :m2] = 1j * self.a_neg * dEdu_hat[:, :, -m1:, :m2]

        # M = |b|²  (dissipative, PSD)
        M_pos = self.b_pos_r**2 + self.b_pos_i**2
        M_neg = self.b_neg_r**2 + self.b_neg_i**2
        diss_hat[:, :, :m1, :m2] = M_pos * dSdu_hat[:, :, :m1, :m2]
        diss_hat[:, :, -m1:, :m2] = M_neg * dSdu_hat[:, :, -m1:, :m2]

        return rev_hat, diss_hat

    # --- single-operator Fourier multipliers (for degeneracy-by-construction) ---
    def _L_apply(self, v, H, W):
        """Apply the skew diagonal operator D_L = i*a to physical field v."""
        m1, m2 = self.m1, self.m2
        vh = torch.fft.rfft2(v, dim=(-2, -1))
        out = torch.zeros_like(vh)
        out[:, :, :m1, :m2] = 1j * self.a_pos * vh[:, :, :m1, :m2]
        out[:, :, -m1:, :m2] = 1j * self.a_neg * vh[:, :, -m1:, :m2]
        return torch.fft.irfft2(out, s=(H, W))

    def _M_apply(self, v, H, W):
        """Apply the PSD diagonal operator D_M = |b|^2 to physical field v."""
        m1, m2 = self.m1, self.m2
        vh = torch.fft.rfft2(v, dim=(-2, -1))
        out = torch.zeros_like(vh)
        Mp = self.b_pos_r**2 + self.b_pos_i**2
        Mn = self.b_neg_r**2 + self.b_neg_i**2
        out[:, :, :m1, :m2] = Mp * vh[:, :, :m1, :m2]
        out[:, :, -m1:, :m2] = Mn * vh[:, :, -m1:, :m2]
        return torch.fft.irfft2(out, s=(H, W))

    @staticmethod
    def _remove(v, w):
        """(I - P_w) v: remove the component of v along direction w, per sample.
        Scale-invariant in w (ratio), so the L2-vs-Euclidean choice is irrelevant."""
        ip = (v * w).sum(dim=(-1, -2), keepdim=True)
        nn = (w * w).sum(dim=(-1, -2), keepdim=True) + 1e-12
        return v - (ip / nn) * w

    def _generic_rhs(self, dEdu, dSdu, H, W):
        """du/dt = (I-P_S) D_L (I-P_S) dE + (I-P_E) D_M (I-P_E) dS.
        Degeneracy (L dS = 0, M dE = 0) holds exactly => dE/dt = 0 and
        dS/dt = <dS, M dS> >= 0 by construction, no projection needed."""
        rev = self._remove(self._L_apply(self._remove(dEdu, dSdu), H, W), dSdu)
        diss = self._remove(self._M_apply(self._remove(dSdu, dEdu), H, W), dEdu)
        return rev, diss

    def _field(self, u, H, W):
        """Reversible+dissipative increment rev+diss at state u (degeneracy path).
        u must require grad (RK4 intermediate states do). create_graph follows
        training so gradients flow through the integrator stages when training and
        eval stays memory-light."""
        cg = self.training
        E = self.E_net(u)
        S = self.S_net(u)
        dEdu = torch.autograd.grad(E.sum(), u, create_graph=cg)[0]
        dSdu = torch.autograd.grad(S.sum(), u, create_graph=cg)[0]
        if self.l2_vargrad:
            scale = (H * W) / (2.0 * math.pi) ** 2
            dEdu = dEdu * scale
            dSdu = dSdu * scale
        rev, diss = self._generic_rhs(dEdu, dSdu, H, W)
        return rev + diss

    def _rk4_step(self, u, H, W):
        """4th-order Runge-Kutta on the learned increment field. RK4's stability
        region contains a segment of the imaginary axis, so it suppresses the
        explicit-Euler amplitude growth of the skew (reversible) operator and
        tightens the finite-step energy drift to O(dt^5). Degeneracy
        (<dE,f>=0, <dS,Mf>>=0) holds at every stage, so the structural guarantees
        are unchanged; only the integration of them improves."""
        u0 = u.detach().requires_grad_(True)
        k1 = self._field(u0, H, W)
        k2 = self._field(u0 + 0.5 * k1, H, W)
        k3 = self._field(u0 + 0.5 * k2, H, W)
        k4 = self._field(u0 + k3, H, W)
        dudt = (k1 + 2.0 * k2 + 2.0 * k3 + k4) / 6.0
        return u + dudt

    def forward(self, u):
        B, C, H, W = u.shape

        if self.degeneracy_construction and self.integrator == 'rk4':
            return self._rk4_step(u, H, W)

        u_leaf = u.detach().requires_grad_(True)

        E = self.E_net(u_leaf)
        S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]

        if self.l2_vargrad:
            # L2 variational derivative: delta E/delta u = (N/|Omega|) grad_u E.
            scale = (H * W) / (2.0 * math.pi) ** 2
            dEdu = dEdu * scale
            dSdu = dSdu * scale

        if self.degeneracy_construction:
            # Thermodynamically-consistent path: degeneracy by construction (Euler).
            rev, diss = self._generic_rhs(dEdu, dSdu, H, W)
            dudt = rev + diss
            return u + dudt

        # --- legacy projection + correction path (ablation only) ---
        dEdu_hat = torch.fft.rfft2(dEdu, dim=(-2, -1))
        dSdu_hat = torch.fft.rfft2(dSdu, dim=(-2, -1))

        rev_hat, diss_hat = self._apply_operators(dEdu_hat, dSdu_hat, H, W)
        rev = torch.fft.irfft2(rev_hat, s=(H, W))
        diss = torch.fft.irfft2(diss_hat, s=(H, W))

        dudt = rev + diss
        dudt = self._project_energy_conservation(dudt, dEdu)
        dudt = self._ensure_entropy_production(dudt, dSdu, dEdu)

        if self.use_residual:
            gate = torch.sigmoid(self.residual_gate)
            residual = gate * self.residual(u_leaf)
            residual = self._project_energy_conservation(residual, dEdu)
            dudt = dudt + residual

        return u + dudt

    def _project_energy_conservation(self, dudt, dEdu):
        """Project du/dt ⊥ δE/δu over both spatial dims → dE/dt = 0."""
        inner = (dudt * dEdu).sum(dim=(-1, -2), keepdim=True)
        norm_sq = (dEdu * dEdu).sum(dim=(-1, -2), keepdim=True) + 1e-10
        return dudt - (inner / norm_sq) * dEdu

    def _ensure_entropy_production(self, dudt, dSdu, dEdu):
        """Ensure dS/dt = ⟨δS/δu, du/dt⟩ ≥ 0 via correction ⊥ δE/δu."""
        dSdt = (dSdu * dudt).sum(dim=(-1, -2), keepdim=True)
        violation = F.relu(-dSdt)
        if violation.sum() > 0:
            inner_SE = (dSdu * dEdu).sum(dim=(-1, -2), keepdim=True)
            norm_E_sq = (dEdu * dEdu).sum(dim=(-1, -2), keepdim=True) + 1e-10
            dSdu_perp = dSdu - (inner_SE / norm_E_sq) * dEdu
            inner_S_Sperp = (dSdu * dSdu_perp).sum(dim=(-1, -2), keepdim=True) + 1e-10
            alpha = violation / inner_S_Sperp
            dudt = dudt + alpha * dSdu_perp
        return dudt

    def entropy_production(self, u):
        """Normalized entropy production r_S = <dS/du, du/dt> / (||dS/du|| ||du/dt||),
        fully differentiable. Used as a minimum-entropy-production (MEP) penalty:
        among GENERIC representations consistent with the data, prefer the one that
        produces the least entropy. Scale-invariant in S (can't be gamed by
        rescaling S), so reducing it requires genuinely making du/dt more
        orthogonal to dS/du -- i.e. genuinely less dissipative."""
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf); S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        H, W = u.shape[-2], u.shape[-1]
        if self.l2_vargrad:
            scale = (H * W) / (2.0 * math.pi) ** 2
            dEdu = dEdu * scale
            dSdu = dSdu * scale
        if self.degeneracy_construction:
            rev, diss = self._generic_rhs(dEdu, dSdu, H, W)
            dudt = rev + diss
        else:
            dEh = torch.fft.rfft2(dEdu, dim=(-2, -1))
            dSh = torch.fft.rfft2(dSdu, dim=(-2, -1))
            rev_hat, diss_hat = self._apply_operators(dEh, dSh, H, W)
            rev = torch.fft.irfft2(rev_hat, s=(H, W))
            diss = torch.fft.irfft2(diss_hat, s=(H, W))
            dudt = rev + diss
            dudt = self._project_energy_conservation(dudt, dEdu)
            dudt = self._ensure_entropy_production(dudt, dSdu, dEdu)
            if self.use_residual:
                gate = torch.sigmoid(self.residual_gate)
                res = self._project_energy_conservation(gate * self.residual(u_leaf), dEdu)
                dudt = dudt + res
        num = (dSdu * dudt).flatten(1).sum(dim=1)
        den = dSdu.flatten(1).norm(dim=1) * dudt.flatten(1).norm(dim=1) + 1e-8
        return (num / den).clamp(min=0).mean()

    def predict_with_info(self, u):
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf)
        S = self.S_net(u_leaf)
        _ = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        _ = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        u_next = self.forward(u)
        u_next_leaf = u_next.detach().requires_grad_(True)
        E_next = self.E_net(u_next_leaf)
        S_next = self.S_net(u_next_leaf)
        info = {
            'E': E.detach(), 'S': S.detach(),
            'dE': (E_next - E).detach(), 'dS': (S_next - S).detach(),
        }
        return u_next, info

    def degeneracy_loss(self, u):
        """Soft regularizer: L·δS/δu ≈ 0 and M·δE/δu ≈ 0.
        Under degeneracy_construction these hold exactly, so the penalty is 0
        (kept only for the legacy projection path)."""
        if self.degeneracy_construction:
            return torch.zeros((), device=u.device)
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf)
        S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        dEdu_hat = torch.fft.rfft2(dEdu, dim=(-2, -1))
        dSdu_hat = torch.fft.rfft2(dSdu, dim=(-2, -1))
        m1, m2 = self.m1, self.m2

        # L·δS/δu
        L_dS_pos = 1j * self.a_pos * dSdu_hat[:, :, :m1, :m2]
        L_dS_neg = 1j * self.a_neg * dSdu_hat[:, :, -m1:, :m2]
        loss_L = (L_dS_pos.abs()**2).mean() + (L_dS_neg.abs()**2).mean()

        # M·δE/δu
        M_pos = self.b_pos_r**2 + self.b_pos_i**2
        M_neg = self.b_neg_r**2 + self.b_neg_i**2
        M_dE_pos = M_pos * dEdu_hat[:, :, :m1, :m2]
        M_dE_neg = M_neg * dEdu_hat[:, :, -m1:, :m2]
        loss_M = (M_dE_pos.abs()**2).mean() + (M_dE_neg.abs()**2).mean()

        return loss_L + loss_M


# ============================================================================
# Training
# ============================================================================


# ============================================================================
# Evaluation
# ============================================================================



# ==================== PART B: FIGURES ====================
#!/usr/bin/env python3
# ============================================================================
# generic_fno_figures_v3.py  --  paper figures for GENERIC-FNO
#
# Produces (to FIGDIR, as both .pdf and .png):
#   MAIN TEXT
#     fig_accuracy.pdf       rollout L2 +/- std, per backbone (wave excluded)
#     fig_dissipation.pdf    gauge-invariant r_mech per PDE x backbone
#     fig_resolution.pdf     zero-shot super-resolution: L2 + r_E vs grid
#     fig_interpretability.pdf  E flat / S rising / mechE decaying along rollout
#                               (LIVE: needs torch + model module + checkpoints)
#   APPENDIX
#     fig_degeneracy.pdf     machine-precision degeneracy residuals (log scale)
#     fig_gauge_variance.pdf per-seed rho_M (scatters) vs r_mech (stable)
#     fig_rmech_ordering.pdf r_mech vs ground-truth pi_true (ordering match)
#     fig_euler_rk4.pdf      advection: Euler vs RK4 L2 and r_E vs grid
#
# DATA SOURCE
#   The scalar figures use the VERIFIED seed-averaged numbers from the
#   2026-06-05 three-seed runs (2D FNO, 1D FNO, 1D DeepONet), embedded below.
#   They match the run logs to the digit, so the figures are reproducible with
#   no Drive / pickle dependency.  To regenerate from your own pickles instead,
#   wire up load_from_pkls() (stub at the bottom) and call it in main().
#
#   fig_interpretability is the one figure that needs trajectory arrays; it
#   loads a GENERIC checkpoint and rolls the model out.  It is skipped with a
#   message if torch / the model module / checkpoints are unavailable.
# ============================================================================
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")                       # headless: the usual reason plots "don't work"
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------
BASE   = os.environ.get("GENERIC_FNO_BASE",
                        "/content/drive/MyDrive/GENERIC_FNO_results")
FIGDIR = os.environ.get("GENERIC_FNO_FIGDIR", os.path.join(BASE, "figures"))

PDES   = ["heat", "advection", "burgers"]
PDE_LABEL = {"heat": "Heat", "advection": "Advection", "burgers": "Burgers"}
C = {"FNO": "#7f7f7f", "EP-FNO": "#1f77b4", "GENERIC": "#d62728",
     "DeepONet": "#9467bd", "truth": "#000000",
     "2D FNO": "#d62728", "1D FNO": "#ff7f0e", "DeepONet_bk": "#2ca02c"}

plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 200, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 11, "legend.frameon": False, "axes.axisbelow": True,
})

# ----------------------------------------------------------------------------
# VERIFIED seed-averaged data (mean, std) -- rollout L2
# ----------------------------------------------------------------------------
ACC = {
 "2D FNO": {
   "heat":      {"FNO": (0.113, 0.034), "EP-FNO": (0.111, 0.008), "GENERIC": (0.091, 0.003)},
   "advection": {"FNO": (0.119, 0.004), "EP-FNO": (0.132, 0.006), "GENERIC": (0.179, 0.020)},
   "burgers":   {"FNO": (0.175, 0.025), "EP-FNO": (0.138, 0.011), "GENERIC": (0.026, 0.003)}},
 "1D FNO": {
   "heat":      {"FNO": (0.082, 0.015), "EP-FNO": (0.065, 0.008), "GENERIC": (0.095, 0.012)},
   "advection": {"FNO": (0.118, 0.004), "EP-FNO": (0.114, 0.014), "GENERIC": (0.233, 0.022)},
   "burgers":   {"FNO": (0.073, 0.010), "EP-FNO": (0.056, 0.003), "GENERIC": (0.046, 0.006)}},
 "1D DeepONet": {
   "heat":      {"DeepONet": (0.057, 0.004), "GENERIC": (0.008, 0.001)},
   "advection": {"DeepONet": (0.201, 0.009), "GENERIC": (0.018, 0.002)},
   "burgers":   {"DeepONet": (0.020, 0.002), "GENERIC": (0.016, 0.001)}},
}
MODELS = {"2D FNO": ["FNO", "EP-FNO", "GENERIC"],
          "1D FNO": ["FNO", "EP-FNO", "GENERIC"],
          "1D DeepONet": ["DeepONet", "GENERIC"]}

# gauge-invariant dissipation: (r_mech_mean, r_mech_std, pi_true)
RMECH = {
 "2D FNO":      {"heat": (0.306, 0.114, 2.93e-2), "advection": (0.003, 0.002, 1.39e-7), "burgers": (0.011, 0.004, 1.22e-3)},
 "1D FNO":      {"heat": (0.267, 0.102, 1.70e-2), "advection": (0.029, 0.028, 1.03e-7), "burgers": (0.051, 0.024, 1.57e-3)},
 "1D DeepONet": {"heat": (0.825, 0.019, 1.70e-2), "advection": (0.007, 0.008, 7.42e-8), "burgers": (0.252, 0.048, 1.37e-3)},
}

# zero-shot super-resolution (Euler model of record), trained @64
RES = [64, 96, 128, 192, 256]
SUPERRES = {
 "heat":    {"FNO": [0.112, 0.097, 0.099, 0.091, 0.099],
             "GENERIC": [0.040, 0.026, 0.023, 0.022, 0.023],
             "rE": [3.1e-7, 7.2e-7, 9.7e-7, 1.1e-6, 3.2e-6]},
 "burgers": {"FNO": [0.107, 0.145, 0.131, 0.164, 0.124],
             "GENERIC": [0.033, 0.023, 0.021, 0.019, 0.015],
             "rE": [4.1e-6, 7.7e-6, 1.2e-5, 1.1e-5, 3.7e-5]},
}

# machine-precision verification residuals (16x16 random init), should be ~0
DEGEN = [
 (r"$\langle\delta E, L\,\delta E\rangle$  (skewness)",       7e-19),
 (r"$\langle\delta S, L\,\delta E\rangle$  (entropy degen.)", 1e-15),
 (r"$\langle\delta E, M\,\delta S\rangle$  (energy degen.)",  3e-13),
 (r"$\langle\delta E, \partial_t u\rangle$  (energy cons.)",  3e-13),
 (r"$r_E$ (trained models, rollout)",                          1e-6),
]
MACHINE_EPS = 2.22e-16

# per-seed rho_M (gauge-dependent) and r_mech (gauge-invariant)
RHOM_SEEDS = {
 "2D FNO":      {"heat": [0.9993, 0.9994, 0.9997], "advection": [0.0004, 0.0001, 0.0004], "burgers": [0.0015, 0.9995, 0.9999]},
 "1D FNO":      {"heat": [0.7728, 0.5427, 0.2984], "advection": [0.0033, 0.0062, 0.0133], "burgers": [0.1975, 0.9253, 0.2547]},
 "1D DeepONet": {"heat": [0.9067, 0.9313, 0.8031], "advection": [0.0238, 0.0378, 0.0305], "burgers": [0.3427, 0.5703, 0.6879]},
}
RMECH_SEEDS = {
 "2D FNO":      {"heat": [0.1450, 0.3729, 0.3995], "advection": [0.0011, 0.0063, 0.0015], "burgers": [0.0086, 0.0085, 0.0165]},
 "1D FNO":      {"heat": [0.3532, 0.3246, 0.1236], "advection": [0.0104, 0.0080, 0.0676], "burgers": [0.0232, 0.0818, 0.0466]},
 "1D DeepONet": {"heat": [0.8189, 0.8507, 0.8055], "advection": [0.0176, -0.0010, 0.0030], "burgers": [0.1859, 0.2985, 0.2708]},
}

# advection Euler vs RK4+AA (limitations), trained @64
ERK_RES = [64, 128, 256]
EULER_RK4 = {
 "FNO":        {"L2": [0.174, 0.134, 0.117]},
 "Euler":      {"L2": [0.606, 0.154, 0.161], "rE": [2e-8, 9e-8, 1e-7]},
 "RK4":        {"L2": [0.341, 0.153, 0.159], "rE": [2.9e-3, 1.2e-3, 8e-5]},
}


# ----------------------------------------------------------------------------
# helpers
# ----------------------------------------------------------------------------
def _save(fig, name):
    os.makedirs(FIGDIR, exist_ok=True)
    written = []
    for ext in ("pdf", "png"):
        p = os.path.join(FIGDIR, f"{name}.{ext}")
        try:
            fig.savefig(p, bbox_inches="tight")
            written.append(ext)
        except Exception as e:
            print(f"  [WARN] could not save {p}: {e}")
    plt.close(fig)
    print(f"  saved {name}: {('+'.join(written)) if written else 'NOTHING'}  -> {FIGDIR}")


# ----------------------------------------------------------------------------
# MAIN-TEXT FIGURES
# ----------------------------------------------------------------------------
def fig_accuracy():
    backbones = ["2D FNO", "1D FNO", "1D DeepONet"]
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
    for ax, bk in zip(axes, backbones):
        models = MODELS[bk]
        x = np.arange(len(PDES)); w = 0.8 / len(models)
        for j, m in enumerate(models):
            means = [ACC[bk][p][m][0] for p in PDES]
            stds  = [ACC[bk][p][m][1] for p in PDES]
            ax.bar(x + (j - (len(models) - 1) / 2) * w, means, w, yerr=stds,
                   capsize=3, label=m, color=C[m], edgecolor="black", linewidth=0.4)
        ax.set_yscale("log")
        ax.set_xticks(x); ax.set_xticklabels([PDE_LABEL[p] for p in PDES])
        ax.set_title(bk); ax.set_ylabel(r"rollout $L^2$ (log)")
        ax.grid(axis="y", ls=":", alpha=0.5)
    # one shared legend BELOW the panels so it never overlaps the bars
    handles, labels = [], []
    for ax in axes:
        for h, l in zip(*ax.get_legend_handles_labels()):
            if l not in labels:
                labels.append(l); handles.append(h)
    fig.suptitle("Predictive accuracy (10-step rollout, 3-seed mean$\\pm$std; wave excluded)", y=1.02)
    fig.tight_layout()
    fig.legend(handles, labels, loc="lower center", ncol=len(labels),
               bbox_to_anchor=(0.5, -0.05), fontsize=9)
    _save(fig, "fig_accuracy")


def fig_dissipation():
    backbones = list(RMECH.keys())
    fig, ax = plt.subplots(figsize=(7.4, 3.6))
    x = np.arange(len(PDES)); w = 0.8 / len(backbones)
    bkcol = {"2D FNO": "#d62728", "1D FNO": "#ff7f0e", "1D DeepONet": "#2ca02c"}
    for j, bk in enumerate(backbones):
        means = [RMECH[bk][p][0] for p in PDES]
        stds  = [RMECH[bk][p][1] for p in PDES]
        ax.bar(x + (j - (len(backbones) - 1) / 2) * w, means, w, yerr=stds,
               capsize=3, label=bk, color=bkcol[bk], edgecolor="black", linewidth=0.4)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels([PDE_LABEL[p] for p in PDES])
    ax.set_ylabel(r"gauge-invariant dissipation $r_{\mathrm{mech}}$")
    ax.set_title(r"Reversible advection $\to r_{\mathrm{mech}}\approx 0$;  ordering matches ground truth")
    ax.annotate(r"advection $\approx 0$" + "\n(reversible)", xy=(1, 0.02), xytext=(1.15, 0.45),
                fontsize=8, ha="left", arrowprops=dict(arrowstyle="->", lw=0.7))
    ax.legend(fontsize=8); ax.grid(axis="y", ls=":", alpha=0.5)
    _save(fig, "fig_dissipation")


def fig_resolution():
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    # left: L2 vs resolution
    ax = axes[0]
    for p, ls in (("heat", "-"), ("burgers", "--")):
        ax.plot(RES, SUPERRES[p]["FNO"], ls, color=C["FNO"], marker="o", ms=4,
                label=f"FNO ({p})")
        ax.plot(RES, SUPERRES[p]["GENERIC"], ls, color=C["GENERIC"], marker="s", ms=4,
                label=f"GENERIC ({p})")
    ax.axvline(64, color="black", ls=":", lw=0.8)
    ax.text(64, ax.get_ylim()[1] * 0.92, " train grid", fontsize=8)
    ax.set_xlabel("evaluation resolution"); ax.set_ylabel(r"rollout $L^2$")
    ax.set_title("Zero-shot super-resolution accuracy")
    ax.set_xticks(RES); ax.legend(fontsize=7.5); ax.grid(ls=":", alpha=0.5)
    # right: r_E vs resolution
    ax = axes[1]
    ax.plot(RES, SUPERRES["heat"]["rE"], "-o", ms=4, color="#9467bd", label="heat")
    ax.plot(RES, SUPERRES["burgers"]["rE"], "--s", ms=4, color="#8c564b", label="burgers")
    ax.axvline(64, color="black", ls=":", lw=0.8)
    ax.set_yscale("log"); ax.set_xlabel("evaluation resolution")
    ax.set_ylabel(r"structural residual $r_E$ (log)")
    ax.set_title("Guarantees transfer too ($r_E$ stays negligible)")
    ax.set_xticks(RES); ax.legend(fontsize=8); ax.grid(ls=":", alpha=0.5)
    _save(fig, "fig_resolution")


# ----------------------------------------------------------------------------
# APPENDIX FIGURES
# ----------------------------------------------------------------------------
def fig_degeneracy():
    labels = [d[0] for d in DEGEN][::-1]
    vals   = [d[1] for d in DEGEN][::-1]
    fig, ax = plt.subplots(figsize=(7.6, 3.0))
    y = np.arange(len(labels))
    ax.barh(y, vals, color="#d62728", edgecolor="black", linewidth=0.4, height=0.6)
    ax.axvline(MACHINE_EPS, color="black", ls="--", lw=1.0)
    ax.text(MACHINE_EPS, len(labels) - 0.4, " float64 $\\epsilon$", fontsize=8, rotation=90, va="top")
    ax.set_xscale("log"); ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=8.5)
    ax.set_xlim(1e-20, 1e-4); ax.set_xlabel("residual magnitude (log)")
    ax.set_title("Structural identities hold to machine precision (any init/dim/res)")
    ax.grid(axis="x", ls=":", alpha=0.5)
    _save(fig, "fig_degeneracy")


def fig_gauge_variance():
    backbones = list(RHOM_SEEDS.keys())
    dpdes = ["heat", "burgers"]                       # the non-reversible ones
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharex=True)
    groups = [(bk, p) for bk in backbones for p in dpdes]
    xticklab = [f"{bk}\n{PDE_LABEL[p]}" for (bk, p) in groups]
    bkcol = {"2D FNO": "#d62728", "1D FNO": "#ff7f0e", "1D DeepONet": "#2ca02c"}
    for ax, (data, title, ylab) in zip(
            axes,
            [(RHOM_SEEDS, r"Gauge-DEPENDENT $\rho_M$ (per seed): unstable", r"$\rho_M$"),
             (RMECH_SEEDS, r"Gauge-INVARIANT $r_{\mathrm{mech}}$ (per seed): stable", r"$r_{\mathrm{mech}}$")]):
        for i, (bk, p) in enumerate(groups):
            pts = data[bk][p]
            ax.scatter([i] * len(pts), pts, s=42, color=bkcol[bk],
                       edgecolor="black", linewidth=0.4, zorder=3)
            ax.plot([i, i], [min(pts), max(pts)], color=bkcol[bk], lw=1.0, alpha=0.6)
        ax.set_xticks(range(len(groups))); ax.set_xticklabels(xticklab, fontsize=7)
        ax.set_ylabel(ylab); ax.set_title(title, fontsize=10); ax.grid(axis="y", ls=":", alpha=0.5)
    axes[0].annotate("same flow,\n$\\rho_M$ flips 0$\\leftrightarrow$1", xy=(3, 0.5), xytext=(3.4, 0.5),
                     fontsize=8, arrowprops=dict(arrowstyle="->", lw=0.7))
    fig.suptitle("Gauge freedom: channel attribution scatters across seeds while physical dissipation does not", y=1.04, fontsize=10)
    _save(fig, "fig_gauge_variance")


def fig_rmech_ordering():
    bkcol = {"2D FNO": "#d62728", "1D FNO": "#ff7f0e", "1D DeepONet": "#2ca02c"}
    mk = {"heat": "o", "advection": "^", "burgers": "s"}
    fig, ax = plt.subplots(figsize=(6.2, 4.2))
    for bk in RMECH:
        for p in PDES:
            rm, _, pit = RMECH[bk][p]
            ax.scatter(pit, max(rm, 1e-4), s=70, marker=mk[p], color=bkcol[bk],
                       edgecolor="black", linewidth=0.5, zorder=3)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(r"ground-truth dissipation $\Pi^\star$ (log)")
    ax.set_ylabel(r"model $r_{\mathrm{mech}}$ (log)")
    ax.set_title(r"$r_{\mathrm{mech}}$ recovers the ground-truth dissipation ordering")
    # legends
    from matplotlib.lines import Line2D
    leg1 = [Line2D([], [], marker=mk[p], color="w", markerfacecolor="gray",
                   markeredgecolor="black", ms=8, label=PDE_LABEL[p]) for p in PDES]
    leg2 = [Line2D([], [], marker="o", color="w", markerfacecolor=bkcol[bk],
                   markeredgecolor="black", ms=8, label=bk) for bk in RMECH]
    l1 = ax.legend(handles=leg1, title="PDE", fontsize=8, loc="upper left")
    ax.add_artist(l1); ax.legend(handles=leg2, title="backbone", fontsize=8, loc="lower right")
    ax.grid(ls=":", alpha=0.5)
    _save(fig, "fig_rmech_ordering")


def fig_euler_rk4():
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    ax = axes[0]
    ax.plot(ERK_RES, EULER_RK4["FNO"]["L2"], "-o", ms=5, color=C["FNO"], label="FNO (baseline)")
    ax.plot(ERK_RES, EULER_RK4["Euler"]["L2"], "-s", ms=5, color="#d62728", label="GENERIC (Euler)")
    ax.plot(ERK_RES, EULER_RK4["RK4"]["L2"], "--^", ms=5, color="#1f77b4", label="GENERIC (RK4+AA)")
    ax.axvline(64, color="black", ls=":", lw=0.8); ax.text(64, ax.get_ylim()[1]*0.9, " train grid", fontsize=8)
    ax.set_xlabel("evaluation resolution"); ax.set_ylabel(r"advection rollout $L^2$")
    ax.set_title("RK4 mitigates the coarse-grid blow-up"); ax.set_xticks(ERK_RES)
    ax.legend(fontsize=8); ax.grid(ls=":", alpha=0.5)
    ax = axes[1]
    ax.plot(ERK_RES, EULER_RK4["Euler"]["rE"], "-s", ms=5, color="#d62728", label="Euler ($\\sim$1e-7)")
    ax.plot(ERK_RES, EULER_RK4["RK4"]["rE"], "--^", ms=5, color="#1f77b4", label="RK4 ($\\sim$1e-3)")
    ax.set_yscale("log"); ax.axvline(64, color="black", ls=":", lw=0.8)
    ax.set_xlabel("evaluation resolution"); ax.set_ylabel(r"energy residual $r_E$ (log)")
    ax.set_title("...but RK4 forfeits exact discrete degeneracy"); ax.set_xticks(ERK_RES)
    ax.legend(fontsize=8); ax.grid(ls=":", alpha=0.5)
    fig.suptitle("Why we keep explicit Euler: the machine-precision discrete guarantee", y=1.04, fontsize=10)
    _save(fig, "fig_euler_rk4")


# ----------------------------------------------------------------------------
# INTERPRETABILITY (LIVE) -- SINGLE NOTEBOOK, NO EXTERNAL FILES.
# Uses the GENERIC_FNO2d class and generate_*_data_2d functions ALREADY DEFINED
# in your notebook.  Run your model-definition cell(s) first, then run this
# block as a cell (paste it -- do NOT `import` it -- so it shares the notebook's
# global namespace).  Needs the nx128 GENERIC checkpoints on BASE.
# ----------------------------------------------------------------------------
def fig_interpretability(n_steps=12, n_ic=6, models=None):
    """Roll a TRAINED GENERIC-FNO2d out and plot learned energy E (flat),
    learned entropy S (rising), and fixed mechanical energy Q=0.5||u||^2
    (decays for dissipative PDEs), per PDE.  Pulls GENERIC_FNO2d and the
    generate_*_data_2d functions from the notebook namespace; loads weights from
    the nx128 checkpoints on BASE.  Skipped (never faked) if anything is missing."""
    try:
        import torch
    except Exception as e:
        print(f"  [skip] fig_interpretability: torch not importable ({e}).")
        return
    G = globals()
    Model = G.get("GENERIC_FNO2d")
    GEN = {"heat": G.get("generate_heat_data_2d"),
           "advection": G.get("generate_advection_data_2d"),
           "burgers": G.get("generate_burgers_data_2d")}
    if Model is None or any(v is None for v in GEN.values()):
        miss = ([] if Model else ["GENERIC_FNO2d"]) + [k for k, v in GEN.items() if v is None]
        print(f"  [skip] fig_interpretability: {', '.join(miss)} not in the notebook "
              f"namespace -- run your model-definition cell first, and paste this block "
              f"as a cell rather than importing it.")
        return
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    panels = {}
    for p in PDES:
        try:
            if models and p in models:                  # in-memory trained model: NO files
                model = models[p].to(dev).eval(); nx = int(getattr(model, "nx", 128))
            else:                                        # else load the saved checkpoint
                ck = os.path.join(BASE, f"generic_fno_2d_{p}_nx128.pt")
                if not os.path.exists(ck):
                    print(f"  [skip] {p}: no checkpoint at {ck}; or call "
                          f"fig_interpretability(models={{'{p}': your_trained_model}})"); continue
                state = torch.load(ck, map_location=dev)
                if isinstance(state, dict) and "state_dict" in state:
                    sd, nx = state["state_dict"], int(state.get("nx", 128))
                else:
                    sd = state.get("model", state) if isinstance(state, dict) else state; nx = 128
                model = Model(nx=nx, degeneracy_construction=True).to(dev)
                model.load_state_dict(sd); model.eval()
            di, _, _ = GEN[p](n_samples=n_ic, nx=nx, nt=2, device=dev)    # [B, nt, H, W]
            u = (di if di.dim() == 4 else di.unsqueeze(1))[:, :1].to(dev).float()
            E, S, Q = [], [], []
            for _ in range(n_steps):
                with torch.no_grad():                # E,S,Q are just recorded values
                    E.append(float(model.E_net(u).mean()))
                    S.append(float(model.S_net(u).mean()))
                    Q.append(float(0.5 * (u ** 2).sum(dim=(-1, -2)).mean()))
                u = model(u).detach()                # forward differentiates E,S internally:
                                                     # do NOT wrap in no_grad
            panels[p] = (np.array(E), np.array(S), np.array(Q))
        except Exception as e:
            print(f"  [skip] {p}: {e}")
    if not panels:
        print("  [skip] fig_interpretability: no panels (run where the nx128 checkpoints live).")
        return
    fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 3.2), squeeze=False)
    nrm = lambda a: (a - a.min()) / (np.ptp(a) + 1e-12)
    for ax, (p, (E, S, Q)) in zip(axes[0], panels.items()):
        t = np.arange(len(E))
        ax.plot(t, nrm(E), "-o", ms=3, color="#1f77b4", label=r"$E[u_t]$ (energy)")
        ax.plot(t, nrm(S), "-s", ms=3, color="#d62728", label=r"$S[u_t]$ (entropy)")
        ax.plot(t, nrm(Q), "--", color="black", label=r"$Q=\frac{1}{2}\|u\|^2$")
        ax.set_title(PDE_LABEL[p]); ax.set_xlabel("rollout step"); ax.set_ylabel("normalized")
        ax.legend(fontsize=7); ax.grid(ls=":", alpha=0.5)
    fig.suptitle(r"Learned thermodynamics along rollouts: $E$ conserved, $S$ rises and tracks $Q$ decay",
                 y=1.04, fontsize=10)
    _save(fig, "fig_interpretability")


# ----------------------------------------------------------------------------
# Optional: reload scalar data from your own pickles instead of the embedded
# verified constants.  Fill in to match your pkl schema, then call in main().
# ----------------------------------------------------------------------------
def load_from_pkls(base=BASE):
    """STUB. Return dicts overriding ACC / RMECH / SUPERRES from the newest
    *.pkl in `base`.  Left unimplemented because the embedded constants are
    already the verified seed-averaged values; wire this up only if you re-run."""
    raise NotImplementedError


# ----------------------------------------------------------------------------
def main():
    print(f"FIGDIR = {FIGDIR}")
    print("Main-text figures:")
    fig_accuracy(); fig_dissipation(); fig_resolution()
    fig_interpretability()
    print("Appendix figures:")
    fig_degeneracy(); fig_gauge_variance(); fig_rmech_ordering(); fig_euler_rk4()
    print("done.")


if __name__ == "__main__":

    import os
    from google.colab import drive

    # Check if Drive is already mounted and unmount it if it is
    if os.path.ismount('/content/drive'):
        try:
            drive.flush_and_unmount()
            print("Google Drive unmounted successfully.")
        except Exception as e:
            print(f"Error unmounting Google Drive: {e}")
            print("It might be necessary to restart the Colab runtime if the issue persists.")

    # Now try to mount Drive
    try:
        drive.mount('/content/drive', force_remount=True)
        print("Google Drive mounted successfully.")
    except ValueError as e:
        if "Mountpoint must not already contain files" in str(e):
            print(f"Failed to mount Google Drive: {e}")
            print("This error can occur if '/content/drive' contains unmanaged files. Consider restarting the Colab runtime.")
        else:
            raise e
    except Exception as e:
        print(f"An unexpected error occurred during mounting: {e}")

    save_dir = '/content/drive/MyDrive/GENERIC_FNO_results'
    main()

Mounted at /content/drive
Google Drive mounted successfully.
FIGDIR = /content/drive/MyDrive/GENERIC_FNO_results/figures
Main-text figures:
  saved fig_accuracy: pdf+png  -> /content/drive/MyDrive/GENERIC_FNO_results/figures
  saved fig_dissipation: pdf+png  -> /content/drive/MyDrive/GENERIC_FNO_results/figures
  saved fig_resolution: pdf+png  -> /content/drive/MyDrive/GENERIC_FNO_results/figures
  saved fig_interpretability: pdf+png  -> /content/drive/MyDrive/GENERIC_FNO_results/figures
Appendix figures:
  saved fig_degeneracy: pdf+png  -> /content/drive/MyDrive/GENERIC_FNO_results/figures
  saved fig_gauge_variance: pdf+png  -> /content/drive/MyDrive/GENERIC_FNO_results/figures
  saved fig_rmech_ordering: pdf+png  -> /content/drive/MyDrive/GENERIC_FNO_results/figures
  saved fig_euler_rk4: pdf+png  -> /content/drive/MyDrive/GENERIC_FNO_results/figures
done.
